# Milestone 2: experiments after the first model

`milestone2_training.ipynb` is the first model (v1): fine-tuned MiniLM, test F1 0.906 vs. 0.863 for the pretrained model. This notebook records what we tried next, in order.

**Rule:** every decision here is made on **val**. The test set is used once, in the last section, for the model we pick. Otherwise the test score would no longer be an honest estimate.

Every experiment adds a row to `outputs/milestone2/experiments/log.csv`.

1. Error analysis: what does v1 get wrong on val, and how often are Google's cards themselves wrong?

In [ ]:
import json, os, random, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import sentence_transformers   # import before torch: the other order crashes on some Windows setups (DLL clash)
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'pairs').is_dir())
sys.path.insert(0, str(ROOT))
from src.embeddings import clean_headline

PAIRS, MODELS = ROOT / 'data' / 'pairs', ROOT / 'models'
OUT = ROOT / 'outputs' / 'milestone2' / 'experiments'
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
SEED = 6450

pairs = {s: pd.read_csv(PAIRS / f'{s}.csv', dtype={'timestamp': str}) for s in ['train', 'val', 'test']}

def best_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return float(thresholds[np.argmax(f1[:-1])])

def evaluate(labels, scores, threshold):
    pred = (scores >= threshold).astype(int)
    return {'precision': precision_score(labels, pred, zero_division=0), 'recall': recall_score(labels, pred, zero_division=0),
            'f1': f1_score(labels, pred, zero_division=0), 'accuracy': accuracy_score(labels, pred),
            'roc_auc': roc_auc_score(labels, scores), 'pr_auc': average_precision_score(labels, scores)}

def pair_scores(model, df, batch_size=64):
    unique = list(dict.fromkeys(pd.concat([df.headline_a, df.headline_b])))
    index = {h: i for i, h in enumerate(unique)}
    E = model.encode([clean_headline(h) for h in unique], batch_size=batch_size, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    return (E[df.headline_a.map(index).values] * E[df.headline_b.map(index).values]).sum(axis=1)

print('device:', DEVICE, '|', {s: len(d) for s, d in pairs.items()})

## 1. Error analysis on val

Score every val pair with v1, use v1's val threshold, and list the mistakes:
- **false positive:** v1 says "same story", but the headlines were in different Google cards;
- **false negative:** v1 says "different story", but they were in the same card.

A random sample of 30 of each is saved to `val_errors_sample.csv` to be read by a person. The column `google_wrong` records whether the headlines are in fact the same story (for a false positive) or different stories (for a false negative), meaning the label is wrong rather than the model.

In [ ]:
v1 = SentenceTransformer(str(MODELS / 'minilm_finetuned'), device=DEVICE)
val = pairs['val'].copy()
val['score'] = pair_scores(v1, val)
t_v1 = best_threshold(val.label.values, val.score.values)
val['pred'] = (val.score >= t_v1).astype(int)
m = evaluate(val.label.values, val.score.values, t_v1)
print(f"v1 on val: threshold {t_v1:.3f}, F1 {m['f1']:.3f}, precision {m['precision']:.3f}, recall {m['recall']:.3f}")

val['error'] = np.select([(val.pred == 1) & (val.label == 0), (val.pred == 0) & (val.label == 1)],
                         ['false positive', 'false negative'], 'correct')
print(val.groupby(['error', 'edition']).size().unstack(fill_value=0))

sample = (val[val.error != 'correct'].groupby('error', group_keys=False)
             .apply(lambda d: d.sample(min(30, len(d)), random_state=SEED)))
cols = ['error', 'edition', 'score', 'headline_a', 'outlet_a', 'headline_b', 'outlet_b', 'timestamp']
sample = sample[cols].assign(score=lambda d: d.score.round(3), google_wrong='', note='')
path = OUT / 'val_errors_sample.csv'
if path.exists():              # never overwrite the hand-checked file
    print('kept existing', path.relative_to(ROOT))
else:
    sample.to_csv(path, index=False)
    print('saved', path.relative_to(ROOT))